# NLLB-200 번역 서비스 — 중간 검증과 실행

이 노트북은 Day 8의 권장 순서를 따른다. **모델 단독 검증 → FastAPI 실행·테스트 → Streamlit UI 확인** 순서로 실행한다.

- 모델: `facebook/nllb-200-distilled-600M`
- 지원 언어: 한국어(`ko`), 영어(`en`), 간체 중국어(`zh`), 일본어(`ja`)
- API: `POST /predict`, Google OAuth/OIDC Bearer 인증

> NLLB는 공개 모델이며 `HF_TOKEN` 없이 실행할 수 있다. 이 노트북은 프로젝트 전용 가상환경 커널에서 실행해야 한다.


## 0. 한 번만 하는 환경 준비

터미널에서 프로젝트 폴더를 연 뒤 아래를 실행한다. `uv`가 설치되어 있지 않다면 먼저 설치해야 한다.

```powershell
uv sync --group dev
Copy-Item .env.example .env
Copy-Item .streamlit/secrets.toml.example .streamlit/secrets.toml
# .env와 .streamlit/secrets.toml에 Google OAuth Client ID를 같은 값으로 설정
# Streamlit은 .env의 TRANSLATION_API_URL에 있는 신뢰된 API로만 ID 토큰을 보낸다
# .env에서 CUDA_VISIBLE_DEVICES=0인지 확인
uv run python -m ipykernel install --user --name translate-project --display-name "Python (translate-project)"
```

그 다음 Jupyter에서 `Python (translate-project)` 커널을 선택한다.


In [14]:
# 프로젝트 루트와 실행 환경을 확인한다.
from pathlib import Path
import os
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app').is_dir():
    raise RuntimeError('project.ipynb를 translate-project 폴더에서 열어 실행하세요.')

load_dotenv(PROJECT_ROOT / '.env')

print('Python:', sys.executable)
print('Project root:', PROJECT_ROOT)
print('.env exists:', (PROJECT_ROOT / '.env').exists())
print('HF_TOKEN configured:', bool(os.getenv('HF_TOKEN')))
print('Translator model:', os.getenv('TRANSLATOR_MODEL_ID', '(default)'))
print('CUDA_VISIBLE_DEVICES:', os.getenv('CUDA_VISIBLE_DEVICES', '(not set)'))


Python: d:\Codes\sandbox\translate-project\.venv\Scripts\python.exe
Project root: d:\Codes\sandbox\translate-project
.env exists: True
HF_TOKEN configured: True
Translator model: facebook/nllb-200-distilled-600M
CUDA_VISIBLE_DEVICES: 1


## 1. 모델 단독 검증

서버를 만들기 전에 네 언어 조합에서 결과를 반환하는지 확인한다. NLLB는 언어별 토큰(`kor_Hang`, `eng_Latn`, `zho_Hans`, `jpn_Jpan`)으로 번역 방향을 지정한다.


In [15]:
from app.config import get_settings

# config.py loads .env before torch is imported, so CUDA_VISIBLE_DEVICES is honored.
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


settings = get_settings()
MODEL_ID = settings.translator_model_id
load_kwargs = {}
if settings.huggingface_token:
    load_kwargs['token'] = settings.huggingface_token
if torch.cuda.is_available():
    load_kwargs.update(device_map='auto', dtype=torch.float16)
    print('Using CUDA:', torch.cuda.get_device_name(0))
    print('PyTorch CUDA build:', torch.version.cuda)
else:
    load_kwargs['dtype'] = torch.float32
    print('CUDA is unavailable. CPU loading may require substantial memory and be slow.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **load_kwargs)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, **load_kwargs)
if not torch.cuda.is_available():
    model.to('cpu')
model.eval()
print('Loaded:', MODEL_ID)


Using CUDA: NVIDIA TITAN RTX
PyTorch CUDA build: 12.6
Loaded: facebook/nllb-200-distilled-600M


In [2]:
def translate_with_model(text: str, source_language: str, target_language: str) -> str:
    """Run one NLLB translation by selecting source and target language tokens."""
    language_codes = {'ko': 'kor_Hang', 'en': 'eng_Latn', 'zh': 'zho_Hans', 'ja': 'jpn_Jpan'}
    tokenizer.src_lang = language_codes[source_language]
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(language_codes[target_language]),
            max_new_tokens=256,
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

samples = [
    ('오늘은 날씨가 좋습니다.', 'ko', 'en'),
    ('I am learning model serving today.', 'en', 'ko'),
    ('今天天气很好。', 'zh', 'en'),
    ('こんにちは、今日はいい天気です。', 'ja', 'ko'),
]

for text, source, target in samples:
    translated = translate_with_model(text, source, target)
    print(f'[{source} → {target}] {text}')
    print('  ', translated, '\n')


[ko → en] 오늘은 날씨가 좋습니다.
   Today's weather is good. 

[en → ko] I am learning model serving today.
   오늘 모델로 봉사하는 법을 배우고 있습니다. 



모델 단독 검증이 통과하면 아래 서버 단계로 넘어간다. 서버는 같은 모델을 별도 프로세스에서 다시 로드하므로, GPU 메모리를 비운다.


In [ ]:
# 서버가 모델을 다시 로드하기 전에 노트북의 검증 모델을 해제한다.
import gc

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Standalone model objects released.')


## 2. FastAPI 서버 실행

Day 8의 `serve_in_thread()` 방식으로 Uvicorn을 백그라운드 스레드에서 실행한다. 서버 시작 시 `lifespan()`이 NLLB를 한 번 로드한다. 첫 실행은 모델 다운로드 때문에 오래 걸릴 수 있다.


In [10]:
import asyncio
import contextlib
import socket
import sys
import threading
import time

import uvicorn

_SERVERS = {}

def port_open(host: str, port: int) -> bool:
    with contextlib.closing(socket.socket()) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0

def stop_server(port: int = 8000) -> None:
    entry = _SERVERS.pop(port, None)
    if entry is None:
        return
    server, thread = entry
    server.should_exit = True
    thread.join(timeout=10)

def serve_in_thread(app_path: str = 'app.main:app', host: str = '127.0.0.1', port: int = 8000):
    stop_server(port)
    if port_open(host, port):
        raise RuntimeError(f'Port {port} is already in use.')

    config = uvicorn.Config(app_path, host=host, port=port, log_level='info')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    state = {'error': None}

    def run_server():
        try:
            loop = asyncio.SelectorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(server.serve())
        except BaseException as error:
            state['error'] = error

    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)

    for second in range(300):
        if port_open(host, port):
            print(f'Server started: http://{host}:{port}')
            return server
        if not thread.is_alive():
            raise RuntimeError('Server thread stopped.') from state['error']
        if second and second % 20 == 0:
            print(f'... model loading ({second}s elapsed)')
        time.sleep(1)
    raise TimeoutError('The server did not start within 5 minutes.')


In [11]:
server = serve_in_thread()


INFO:     Started server process [36420]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started: http://127.0.0.1:8000


INFO:     127.0.0.1:6074 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:6075 - "POST /predict HTTP/1.1" 401 Unauthorized
INFO:     127.0.0.1:6076 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:6077 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:1765 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:10021 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36420]


## 3. API 검증 — health와 인증 경계

인증된 번역 요청은 Streamlit을 통해 검증한다. 아래 셀은 공개 health와 Bearer 토큰 없는 요청의 401을 검증한다. Swagger UI는 [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)에서 Bearer 인증 스키마를 확인한다.


In [ ]:
import requests

API_URL = 'http://127.0.0.1:8000'

health = requests.get(f'{API_URL}/health', timeout=10)
print('Health:', health.status_code, health.json())

no_token = requests.post(
    f'{API_URL}/predict',
    json={'text': 'Hello', 'source_language': 'en', 'target_language': 'ko'},
    timeout=10,
)
print('No Bearer token:', no_token.status_code, no_token.json())

assert health.status_code == 200
assert no_token.status_code == 401


## 4. Streamlit 실행과 UI 확인

Streamlit은 별도 프로세스로 실행한다. Google 계정으로 로그인한 뒤 네 언어 사이의 서로 다른 번역 방향을 테스트한다. Google OAuth는 iframe에 임베드된 앱을 지원하지 않으므로, 아래 링크를 새 탭에서 연다.


In [16]:
import subprocess
import tempfile

STREAMLIT_PORT = 8501
streamlit_process = None

if port_open('127.0.0.1', STREAMLIT_PORT):
    print(f'Streamlit is already running: http://127.0.0.1:{STREAMLIT_PORT}')
else:
    log_path = Path(tempfile.gettempdir()) / 'translate_project_streamlit.log'
    streamlit_log = open(log_path, 'w', encoding='utf-8')
    streamlit_process = subprocess.Popen(
        [
            sys.executable, '-m', 'streamlit', 'run', 'frontend/app.py',
            '--server.port', str(STREAMLIT_PORT),
            '--server.headless', 'true',
        ],
        stdout=streamlit_log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if port_open('127.0.0.1', STREAMLIT_PORT):
            break
        if streamlit_process.poll() is not None:
            streamlit_log.close()
            raise RuntimeError(log_path.read_text(encoding='utf-8'))
        time.sleep(0.5)
    else:
        raise TimeoutError('Streamlit did not start within 30 seconds.')
    print(f'Streamlit started: http://127.0.0.1:{STREAMLIT_PORT}')


Streamlit started: http://127.0.0.1:8501


In [ ]:
from IPython.display import HTML, display

display(HTML(
    f'<a href="http://127.0.0.1:{STREAMLIT_PORT}" target="_blank">Open Streamlit in a new tab</a>'
))


### UI 테스트 체크리스트

1. Google 로그인 후 한국어 → 영어 번역이 되는가?
2. 중국어(간체) → 일본어 등 새 언어 조합도 번역되는가?
3. 로그아웃 후에는 번역 화면 대신 Google 로그인 버튼이 표시되는가?
4. 빈 텍스트를 보내면 Streamlit이 입력 오류를 표시하는가?
5. 서버를 중지한 뒤 요청하면 연결 오류를 표시하는가?
6. 번역 결과의 복사 버튼과 번역 기록이 동작하는가?
7. 짧은 TXT를 업로드해 번역·다운로드할 수 있는가?
8. `text` 열이 있는 50행 이하 CSV를 업로드해 열 선택·다운로드할 수 있는가?
9. OAuth 로그인 뒤 ID 토큰이 만료된 경우 재로그인 안내가 표시되는가?


## 5. 종료와 회고

노트북을 다시 실행하거나 GPU 메모리를 비우기 전에 아래 셀로 서버와 Streamlit을 종료한다. 제출 전에는 API 테스트 결과, Swagger 화면, UI 번역 결과를 캡처하고 README/회고에 기록한다.


In [18]:
stop_server(8000)
if streamlit_process is not None and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
print('FastAPI and Streamlit stopped.')


FastAPI and Streamlit stopped.
